In [0]:
'''what: Aggregations, KPIs, business metrics
   how: Reading the Silver layer delta files 
   why: To transform transactional data into summary data that will be consumed by business teams
'''

In [0]:
%run "./environmental_variables"

In [0]:
def read_silver(table_name):
    return spark.read.table(config["database"] + table_name)
sales_tran_data = read_silver("sales_tran_data")

In [0]:
'''This insights give us Top5 best selling products for each country
'''
from pyspark.sql.functions import *
from pyspark.sql.window import Window
sales_by_country = read_silver("sales_tran_data").groupBy("tran_month",col('franchise_country').alias('country'), ('product')).agg(
        sum("quantity").alias("total_quantity"),
        sum("totalprice").alias("total_revenue")
    ).withColumn('top_5_sales_product',rank().over(Window.partitionBy('country').orderBy(desc('total_revenue'))))\
        .filter(col('top_5_sales_product') <= 5)
 

# sales_by_country.orderBy(col('country')).show(100)

sales_by_country.write.format("delta").mode("overwrite").option("mergeSchema","True").partitionBy('tran_month')\
    .saveAsTable(f"{config['database']}sales_by_country")
print("gold layer table load for sales_by_country")

In [0]:
"""This metrics used to explain the sales trend of each franchise,
    sales comparision between previous and current day , count of sales increase , descrease,
    same for comparision time period.(same can be done for comparing month and year). 

    collect list of date or month or years with comparision which give more insights
    on specific time period for high or low sales.

    "This metrics help us to understand the time period for high and low sales by which,
     we can give insights to the Franchise to plan the inventory"

"""
franchise_performance = (
    read_silver("sales_tran_data")
    .groupBy(to_date(col("datetime")).alias('tran_date'),"franchiseid","franchise_name")
    .agg(
        sum("totalprice").alias("total_sales"),
        countDistinct("customerid").alias("unique_customers"),
        avg("totalprice").alias("avg_transaction_value")
    )
)
sales_comparision=franchise_performance.withColumn("prev_day_sales",lag("total_sales",1,0)\
    .over(Window.partitionBy("franchiseid").orderBy("tran_date")))\
        .withColumn('sales_comparision',when(col('total_sales') > col('prev_day_sales'),'sales increased')\
                                         .when(col('total_sales') < col('prev_day_sales'),'sales decreased')\
                                         .otherwise('sales same'))\
        .withColumn('tran_month',trunc(col('tran_date'),'month'))

sales_comparision_df = sales_comparision.groupBy("tran_month","franchiseid","franchise_name","sales_comparision")\
                                        .agg(collect_list(struct(col('tran_date'))).alias('date_lst'),count("sales_comparision").alias("sales_comparision_cnt"),
                                             sum("total_sales").alias("total_sales"))

# sales_comparision_df.orderBy("franchiseid").show(50)
# franchise_performance.write.format("delta").mode("overwrite").save(config["gold_path"] + "franchise_performance")
sales_comparision_df.write.format("delta").option("mergerSchema","true").mode("overwrite").partitionBy('tran_month').saveAsTable(f"{config['database']}franchise_sales_comparision")
print("gold layer table load for franchise_performance")

In [0]:
"""Insights on review sentiment set by the customers on each Franchise.
   Based on reviews, it has segregate into 3 category, "positive","nuetral","negative" 
   This will ensue Franchise to consider the mood of the customers and based on that they can take actions to improve 
   the customer experience."""

positive_words = ["good", "fresh", "tasty", "excellent", "friendly","delightful",'glad','stumbled']
negative_words = ["bad", "stale", "late", "poor", "rude",'disappoint','negative']

def review_sentiment(text):
    text = text.lower()
    if any(w in text for w in positive_words):
        return "positive"
    elif any(w in text for w in negative_words):
        return "negative"
    else:
        return "neutral"

sentiment_udf = udf(review_sentiment)

reviews_with_sentiment = (
    read_silver("sales_tran_data")
    .withColumn("sentiment", sentiment_udf("clean_review"))
)

review_sentiment = (
    reviews_with_sentiment
    .groupBy("tran_month","franchiseid","franchise_name", "sentiment")
    .count()
)

review_sentiment.write.format("delta").mode("overwrite").option("mergerSchema","true").partitionBy('tran_month').saveAsTable(f"{config['database']}review_sentiment")
print("gold table lead for review_sentiment")

In [0]:
"""This insights on review sentiment set by the customers on each Franchise product.
   So Franchise can understand the customer reviews based on specific product and can make appropiate action to
   satisfy customers
"""
positive_words = ["good", "fresh", "tasty", "excellent", "friendly","delightful",'glad','stumbled']
negative_words = ["bad", "stale", "late", "poor", "rude",'disappoint','negative']

def review_sentiment(text):
    text = text.lower()
    if any(w in text for w in positive_words):
        return "positive"
    elif any(w in text for w in negative_words):
        return "negative"
    else:
        return "neutral"

sentiment_udf = udf(review_sentiment)

reviews_with_sentiment = (
    read_silver("sales_tran_data")
    .withColumn("sentiment", sentiment_udf("clean_review"))
)
review_sentiment = (
    reviews_with_sentiment
    .groupBy("tran_month","franchiseid", "sentiment")
    .count()
)



product_lst =     read_silver("sales_tran_data").select('product').distinct()\
    .withColumn('product_split',split(col('product'),' '))\
        .select(col('product_split')).agg(collect_list('product_split').alias('product_lst')).withColumn('flat_product',flatten("product_lst")).select('flat_product').first()[0]
    
# print(product_lst)
def product_word(text):
    text = text.lower()
    if any(w.lower()  in text for w in product_lst):
        return "yes"
    
    
product_udf = udf(product_word)
product_review_with_sentiment =  reviews_with_sentiment\
        .withColumn("prod_review", product_udf("clean_review"))\
        .filter(col("prod_review").isNotNull())

prod_review_sentiment = product_review_with_sentiment\
                        .groupBy("tran_month","franchiseid",'product', "sentiment")\
    .count()

# prod_review_sentiment.show(100)
review_sentiment.write.format("delta").mode("overwrite").option("mergerSchema","True").partitionBy('tran_month').saveAsTable(f"{config['database']}product_review_sentiment")
print("gold table lead for review_sentiment")